In [7]:
from minsearch import Index

In [8]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

In [9]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q1. How many lesson pages

## How many lesson pages are in the dataset?

### Answer: 72

In [10]:
len(documents)

72

In [11]:
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
# How does the agentic loop keep calling the model until it stops?
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [12]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query)

# Q2. Indexing and searching

## Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

    How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

### Answer: 01-agentic-rag/lessons/14-agentic-loop.md

In [13]:
results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [14]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-18 17:28:31--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-18 17:28:31 (30.5 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [15]:
from openai import OpenAI

client = OpenAI()  # usa OPENAI_API_KEY del environment

In [18]:
from rag_helper import RAGBase

In [19]:
class RAGMarkdown(RAGBase):

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        print(f"Input tokens: {response.usage.input_tokens}")
        print(f"Output tokens: {response.usage.output_tokens}")

        return response.output_text

# Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

    Implement the RAG flow yourself
    Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context

Build a RAG over the index from Q2 and answer the query:

    How does the agentic loop keep calling the model until it stops?

## Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

### Answer: ≈ 7000

In [43]:
rag = RAGMarkdown(index=index, llm_client=client, model='gpt-5.4-mini')
answer = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Input tokens: 7126
Output tokens: 148
The loop keeps calling the model by checking whether the model returned any `function_call` items.

- If there **is** a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are **no** function calls, it breaks out of the `while True` loop.

In the lesson code, this is done with the `has_function_calls` flag:

```python
has_function_calls = False
...
if item.type == "function_call":
    ...
    has_function_calls = True

...
if has_function_calls == False:
    break
```

So the agent keeps going until the model stops asking for tools and returns a final answer.


In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

# Q4. Chunking

chunks = chunk_documents(documents, size=2000, step=1000)

With size=2000 and step=1000 (you can see the implementation here):

How many chunks do you get?

### Answer: 295

In [22]:
len(chunks)

295

In [23]:
import minsearch

# Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

### Answer: 3× fewer

In [24]:
# chunks indexing
chunk_index = minsearch.Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
chunk_index.fit(chunks)

# RAG with chunks
rag_chunked = RAGMarkdown(index=chunk_index, llm_client=client, model='gpt-4o-mini')
answer_chunked = rag_chunked.rag("How does the agentic loop keep calling the model until it stops?")
print(answer_chunked)

Input tokens: 2310
Output tokens: 104
The agentic loop continues to call the model in a while loop structure until the model responds without any function calls. This means that the loop keeps executing until the model has no further tool calls (function calls) to make. The iteration counter tracks how many times this process happens. If the model generates any function calls, the loop will execute again, allowing the model to make further searches or actions based on the previous response. The loop exits when the model finally produces an output that does not involve any function calls.


In [25]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [35]:
def search(query: str) -> dict[str, str]:
    """
    Search the course materials for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5
    )

In [36]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [38]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [39]:
homework_prompt = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [40]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=homework_prompt,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

# Q6. Turning it into an agent

How many times did the agent call search?

### Answer: ≈4

In [42]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
